In [ ]:
from random import randrange, choice
def req(method, id, val):
    o = [method]
    if method not in ('ls','du'):
        o.append(id)
        if method=='set' or method.startswith('math'):
            o.append(val)
    return ' '.join(map(str, o))
def rsp(s, method, id, val, ring):
    t = s.get(id)
    if isinstance(val, str):
        val = s.get(val)
    if method == 'ls':
        return 'ok' + ''.join(f' {i} {j}'for i,j in s.items()if j is not None)
    elif method == 'du':
        return f'ok {len(s)}'
    elif method == 'echo':
        return f'ok {id}'
    elif method == 'get':
        val = t
    elif method == 'set':
        s[id] = val
    elif method == 'del':
        s[id] = val = None
    elif method.startswith('math'):
        if t is None or val is None:
            s[id] = t = None
        elif method == 'mathadd':
            s[id] = t = (t + val) % ring
        elif method == 'mathsub':
            s[id] = t = (t - val) % ring
        elif method == 'mathmul':
            s[id] = t = (t * val) % ring
        elif val == 0:
            s[id] = t = float('nan')
        elif method == 'mathdiv':
            s[id] = t = t // val
        elif method == 'mathmod':
            s[id] = t = t % val
        else:
            return f'nocmd {method}'
    else:
        return f'nocmd {method}'
    return f'ok {"nul"if t is None else t} {"nul"if val is None else val}'
def dat(n=1, m=1, var='abc', ring=10):
    A = []
    B = []
    s = [{}for m in range(m)]
    for n in range(n):
        a = []
        b = []
        for m in s:
            method = choice(('ls','du','echo','get',*('set',)*10,'del','mathadd','mathsub','mathmul','mathdiv','mathmod'))
            id = choice(var)
            val = randrange(ring) #if randrange(2) else choice(var)
            a.append(req(method, id, val))
            b.append(rsp(m, method, id, val, ring))
        A.append(a)
        B.append(b)
    return A, B
def strs(a):
    if isinstance(a, str):
        yield a
    else:
        for a in a:
            yield from strs(a)
def char(a):
    for a in strs(a):
        yield from a
def maxlen(a):
    return max(map(len, strs(a)))

In [ ]:
import torch
from torch.nn import *
class Lang:
    def __init__(s, a):
        s.a = 'EOF',*set(a)
        s.A = {a:i for i,a in enumerate(s.a)}
    def __len__(s):
        return len(s.a)
    def __call__(s, a):
        return torch.tensor([[[s.A[a[i]]if i<len(a)else 0for a in a]for a in a]for i in range(maxlen(a)+1)], device='cuda')
    def __getitem__(s, a):
        o = []
        for a in a.permute(1,2,0).tolist():
            T = []
            for a in a:
                t = []
                for a in a:
                    if a==0:
                        break
                    t.append(s.a[a])
                T.append(''.join(t))
            o.append(T)
        return o
class Seq2Seq(Module):
    def __init__(s, a, x=4, y=32, z=128):
        super().__init__()
        s.a = Lang(a)
        s.emb = Embedding(len(s.a), x)
        s.enc = GRU(x, y)
        s.l = Sequential(Linear(y, y), ReLU(1), Linear(y, y))
        s.rnn = GRU(y, z)
        s.r = Sequential(Linear(z, y), ReLU(1), Linear(y, y))
        s.dec = GRU(1, y)
        s.deb = Sequential(Linear(y, x), ReLU(1), Linear(x, len(s.a)))
        s.cuda()
    def forward(s, x, y=65):
        x = s.a(x)
        x = s.emb(x)
        x = s.enc(x.flatten(1,2))[1][0].unflatten(0, x.shape[1:3])
        x = s.l(x)
        x = s.rnn(x)[0]
        x = s.r(x)
        if isinstance(y, list):
            y = maxlen(y)
        y = torch.linspace(0,1,y,device='cuda')[:,None,None].broadcast_to(y,x.size(0)*x.size(1),1)
        x = s.dec(y, x.flatten(0,1)[None])[0].unflatten(1, x.shape[:2])
        x = s.deb(x)
        return x
    def score(s, x, y, **_):
        y = s.a(y)
        x = s(x, len(y), **_)
        return functional.cross_entropy(x.permute(0,3,1,2), y)
    def predict(s, x, y=65, **_):
        return s.a[s(x,y,**_).argmax(3)]

In [ ]:
from tabulate import tabulate
from IPython.display import HTML
def box(a):
    display(HTML(tabulate({i:list(strs(a))for i,a in a.items()},'keys','html').replace('>\n','>').replace('\n','<br>')))

In [ ]:
m = Seq2Seq(char(dat(999)))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .001, 100000)
for i in range(10):
    for i in range(10):
        for i in range(O.total_steps//100):
            i = m.score(*dat(32, 32))
            i.backward()
            o.step()
            o.zero_grad()
            O.step()
        print(end=f'{i:.3f} ')
    i,j = dat(8, 1)
    with torch.no_grad():
        box({'x':i, 't':j, 'y':m.predict(i,j)})